# Provisioning OpenRouter API keys for BSSDH 2026

This instructor-only notebook provisions one OpenRouter API key for each unique participant email address.

Defaults:

- input: `temp/participants_emails.txt`
- spending limit: USD 1 per key, with no recurring reset
- expiration: exactly seven days after the provisioning run starts, calculated in UTC
- key name: `BSSDH_2026_participant_<sanitized_email>`

The notebook performs a local preflight first. No OpenRouter request is made unless `CONFIRM_PROVISIONING` is explicitly changed to `True`. Plaintext API keys are returned only once by OpenRouter, so successful responses are appended to the private result CSV immediately.


In [ ]:
import csv
import hashlib
import os
import re
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import requests

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **_kwargs):
        return iterable


def find_repository_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".gitignore").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the workshop repository root.")


REPO_ROOT = find_repository_root()
DEFAULT_INPUT_PATH = REPO_ROOT / "temp" / "participants_emails.txt"
DEFAULT_OUTPUT_DIR = REPO_ROOT / "temp"
print(f"Repository root: {REPO_ROOT}")


## Configuration

Leave `INPUT_PATH` and `OUTPUT_DIR` as `None` to use the private repository `temp` directory. An alternate relative input path is resolved from the repository root; an absolute path is also accepted.

Keep `CONFIRM_PROVISIONING = False` while reviewing the preflight. Change it to `True` only when the participant count, total allocation, names, and output locations are correct.


In [ ]:
INPUT_PATH = None       # Example: REPO_ROOT / "temp" / "another_email_list.txt"
OUTPUT_DIR = None       # Defaults to REPO_ROOT / "temp"
LIMIT_USD = 1.0
VALIDITY_DAYS = 7
NAME_PREFIX = "BSSDH_2026_participant"
REQUEST_DELAY_SECONDS = 0.1
REQUEST_TIMEOUT_SECONDS = 30
RESUME_EXISTING = False
CONFIRM_PROVISIONING = False


In [ ]:
EMAIL_SHAPE = re.compile(r"^[^\s@]+@[^\s@]+\.[^\s@]+$")
SAFE_NAME_CHARACTERS = re.compile(r"[^A-Za-z0-9._-]+")


def resolve_path(path, default):
    if path is None:
        return default.resolve()
    candidate = Path(path).expanduser()
    if not candidate.is_absolute():
        candidate = REPO_ROOT / candidate
    return candidate.resolve()


def sanitize_email_for_key_name(email):
    # OpenRouter currently documents no restriction beyond a non-empty name.
    # This conservative slug remains portable if stricter validation is introduced.
    slug = SAFE_NAME_CHARACTERS.sub("_", email.casefold())
    slug = re.sub(r"_+", "_", slug).strip("._-")
    return slug or "email"


def add_unique_key_names(participants, name_prefix):
    used_names = {}
    for participant in participants:
        normalized_email = participant["normalized_email"]
        base_name = f"{name_prefix}_{sanitize_email_for_key_name(normalized_email)}"
        key_name = base_name
        if key_name in used_names and used_names[key_name] != normalized_email:
            digest = hashlib.sha256(normalized_email.encode("utf-8")).hexdigest()[:8]
            key_name = f"{base_name}_{digest}"
            counter = 2
            while key_name in used_names and used_names[key_name] != normalized_email:
                key_name = f"{base_name}_{digest}_{counter}"
                counter += 1
        used_names[key_name] = normalized_email
        participant["key_name"] = key_name
    return participants


def load_participants(input_path=None, name_prefix="BSSDH_2026_participant"):
    path = resolve_path(input_path, DEFAULT_INPUT_PATH)
    if not path.is_file():
        raise FileNotFoundError(f"Participant email file not found: {path}")

    first_by_normalized_email = {}
    duplicate_lines = {}
    invalid_rows = []

    for line_number, raw_line in enumerate(
        path.read_text(encoding="utf-8-sig").splitlines(), start=1
    ):
        email = raw_line.strip()
        if not email:
            continue
        if not EMAIL_SHAPE.fullmatch(email):
            invalid_rows.append({"line": line_number, "value": email})
            continue

        normalized_email = email.casefold()
        if normalized_email in first_by_normalized_email:
            duplicate_lines.setdefault(normalized_email, []).append(line_number)
            continue

        first_by_normalized_email[normalized_email] = {
            "email": email,
            "normalized_email": normalized_email,
            "source_line": line_number,
        }

    participants = add_unique_key_names(
        list(first_by_normalized_email.values()), name_prefix
    )
    duplicate_report = []
    for normalized_email, skipped_lines in duplicate_lines.items():
        kept = first_by_normalized_email[normalized_email]
        duplicate_report.append({
            "email": kept["email"],
            "kept_line": kept["source_line"],
            "skipped_lines": ";".join(str(line) for line in skipped_lines),
            "occurrence_count": 1 + len(skipped_lines),
        })

    return path, participants, duplicate_report, invalid_rows


def write_csv(path, fieldnames, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


## Local preflight

This cell reads the input, skips exact case-insensitive duplicates, creates the duplicate report, validates key-name uniqueness, and prints only aggregate information. It does not contact OpenRouter. Invalid addresses are written to a separate private report and stop the run.


In [ ]:
output_dir = resolve_path(OUTPUT_DIR, DEFAULT_OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
input_file, participants, duplicate_rows, invalid_rows = load_participants(
    INPUT_PATH, NAME_PREFIX
)

duplicate_report_path = output_dir / "BSSDH_2026_duplicate_emails.csv"
invalid_report_path = output_dir / "BSSDH_2026_invalid_emails.csv"
result_path = output_dir / "BSSDH_2026_provisioned_keys.csv"

write_csv(
    duplicate_report_path,
    ["email", "kept_line", "skipped_lines", "occurrence_count"],
    duplicate_rows,
)
write_csv(invalid_report_path, ["line", "value"], invalid_rows)

if invalid_rows:
    raise ValueError(
        f"Found {len(invalid_rows)} invalid email entries. "
        f"Review {invalid_report_path} before provisioning."
    )

key_names = [participant["key_name"] for participant in participants]
if len(key_names) != len(set(key_names)):
    raise ValueError("Sanitized key names are not unique.")
if LIMIT_USD <= 0:
    raise ValueError("LIMIT_USD must be greater than zero.")
if VALIDITY_DAYS <= 0:
    raise ValueError("VALIDITY_DAYS must be greater than zero.")

duplicate_count = sum(row["occurrence_count"] - 1 for row in duplicate_rows)
print(f"Input file: {input_file}")
print(f"Unique participant emails: {len(participants)}")
print(f"Duplicate entries skipped: {duplicate_count}")
print(f"Duplicate report: {duplicate_report_path}")
print(f"Invalid entries: {len(invalid_rows)}")
print(f"Limit per key: USD {LIMIT_USD:.2f}")
print(f"Maximum aggregate allocation: USD {len(participants) * LIMIT_USD:.2f}")
print(f"Validity from provisioning start: {VALIDITY_DAYS} days")
print(f"Provisioning result file: {result_path}")
print(f"Provisioning confirmed: {CONFIRM_PROVISIONING}")


In [ ]:
RESULT_FIELDS = [
    "email",
    "key_name",
    "api_key",
    "key_hash",
    "limit_usd",
    "created_at_utc",
    "expires_at_utc",
    "status",
    "error",
]


def utc_timestamp(value):
    return (
        value.astimezone(timezone.utc)
        .isoformat(timespec="seconds")
        .replace("+00:00", "Z")
    )


def read_existing_results(path):
    if not path.exists():
        return [], set()
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    uncertain = [row for row in rows if row.get("status") == "uncertain"]
    if uncertain:
        raise RuntimeError(
            "The existing result file contains an uncertain request. Reconcile it "
            "manually before resuming; blindly retrying could create a duplicate key "
            "whose plaintext value was lost."
        )
    completed = {
        row["email"].strip().casefold()
        for row in rows
        if row.get("status") == "success"
    }
    return rows, completed


def create_openrouter_key(management_key, key_name, limit_usd, expires_at_utc):
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/keys",
            headers={
                "Authorization": f"Bearer {management_key}",
                "Content-Type": "application/json",
            },
            json={
                "name": key_name,
                "limit": limit_usd,
                "expires_at": expires_at_utc,
            },
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
    except requests.RequestException as exc:
        # A timeout may occur after the server created the key. Never retry blindly.
        return {
            "status": "uncertain",
            "error": f"{type(exc).__name__}: {exc}",
        }

    if response.status_code != 201:
        # A gateway/server timeout can be ambiguous: creation may have succeeded.
        status = (
            "uncertain"
            if response.status_code == 408 or response.status_code >= 500
            else "failed"
        )
        return {
            "status": status,
            "error": f"HTTP {response.status_code}: {response.text[:500]}",
        }

    try:
        response_data = response.json()
    except ValueError as exc:
        return {
            "status": "uncertain",
            "error": f"HTTP 201 returned invalid JSON: {exc}",
        }

    plaintext_key = response_data.get("key")
    metadata = response_data.get("data") or {}
    if not plaintext_key:
        return {
            "status": "uncertain",
            "error": "OpenRouter returned HTTP 201 without a plaintext key.",
        }

    return {
        "status": "success",
        "error": "",
        "api_key": plaintext_key,
        "key_hash": metadata.get("hash", ""),
        "created_at_utc": metadata.get("created_at", ""),
    }


def provision_participants(
    participants,
    result_path,
    *,
    limit_usd=1.0,
    validity_days=7,
    resume_existing=False,
    confirmed=False,
):
    if not confirmed:
        raise RuntimeError(
            "Provisioning is not confirmed. Set CONFIRM_PROVISIONING = True first."
        )

    management_key = os.getenv("OPEN_ROUTER_BSSDH_PROVISIONER")
    if not management_key:
        raise RuntimeError("OPEN_ROUTER_BSSDH_PROVISIONER is not set.")

    existing_rows, completed_emails = read_existing_results(result_path)
    if existing_rows and not resume_existing:
        raise FileExistsError(
            f"Result file already exists: {result_path}. Review it, then set "
            "RESUME_EXISTING = True if appropriate."
        )

    pending = [
        participant
        for participant in participants
        if participant["normalized_email"] not in completed_emails
    ]
    if not pending:
        print("No participant keys remain to be provisioned.")
        return {
            "success": 0,
            "skipped_existing": len(completed_emails),
            "result_path": result_path,
        }

    run_started_utc = datetime.now(timezone.utc).replace(microsecond=0)
    expires_at_utc = utc_timestamp(
        run_started_utc + timedelta(days=validity_days)
    )
    result_path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not result_path.exists() or result_path.stat().st_size == 0
    success_count = 0

    print(f"Provisioning {len(pending)} participant keys.")
    print(f"Run started (UTC): {utc_timestamp(run_started_utc)}")
    print(f"All new keys expire (UTC): {expires_at_utc}")

    with result_path.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=RESULT_FIELDS)
        if write_header:
            writer.writeheader()
            handle.flush()
            os.fsync(handle.fileno())

        for participant in tqdm(
            pending, total=len(pending), desc="Provisioning keys"
        ):
            api_result = create_openrouter_key(
                management_key,
                participant["key_name"],
                limit_usd,
                expires_at_utc,
            )
            row = {
                "email": participant["email"],
                "key_name": participant["key_name"],
                "api_key": api_result.get("api_key", ""),
                "key_hash": api_result.get("key_hash", ""),
                "limit_usd": f"{limit_usd:.2f}",
                "created_at_utc": api_result.get("created_at_utc", ""),
                "expires_at_utc": expires_at_utc,
                "status": api_result["status"],
                "error": api_result.get("error", ""),
            }
            writer.writerow(row)
            handle.flush()
            os.fsync(handle.fileno())

            if api_result["status"] != "success":
                raise RuntimeError(
                    f"Provisioning stopped after a {api_result['status']} request. "
                    f"Review {result_path}."
                )

            success_count += 1
            if REQUEST_DELAY_SECONDS:
                time.sleep(REQUEST_DELAY_SECONDS)

    return {
        "success": success_count,
        "skipped_existing": len(completed_emails),
        "expires_at_utc": expires_at_utc,
        "result_path": result_path,
    }


## Provision participant keys

This is the notebook's only provisioning call. It creates no test keys. Review the preflight output, change `CONFIRM_PROVISIONING` to `True` in the configuration cell, rerun that cell, and then run this cell.


In [ ]:
if CONFIRM_PROVISIONING:
    provisioning_summary = provision_participants(
        participants,
        result_path,
        limit_usd=LIMIT_USD,
        validity_days=VALIDITY_DAYS,
        resume_existing=RESUME_EXISTING,
        confirmed=CONFIRM_PROVISIONING,
    )
    print("Provisioning complete.")
    print(f"Successful keys in this run: {provisioning_summary['success']}")
    print(
        "Previously completed keys skipped: "
        f"{provisioning_summary['skipped_existing']}"
    )
    if provisioning_summary.get("expires_at_utc"):
        print(
            "New keys expire (UTC): "
            f"{provisioning_summary['expires_at_utc']}"
        )
    print(f"Private result file: {provisioning_summary['result_path']}")
else:
    print("Preflight only: no OpenRouter keys were created.")
    print(
        "Set CONFIRM_PROVISIONING = True in the configuration cell when ready."
    )
